# 05 — VCOD setup, dataset validation, and backbone mapping gate
Run this notebook from a fresh A100-class Colab GPU runtime. It reuses the repository bootstrap, validates the canonical MoCA-Mask and CamoVid60K manifests, records overlap and manual-review receipts in Drive, and verifies the exact DINOv3/V-JEPA2.1 checkpoint pathways before training.

In [ ]:
# Fresh-kernel bootstrap. Edit only these settings; cached Drive assets are reused.
PROJECT_REPO_URL = 'https://github.com/papanag/cod-ssl.git'
PROJECT_BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/cod-ssl'

from google.colab import drive
drive.mount('/content/drive')
from getpass import getpass
from pathlib import Path
import json, os, subprocess, sys, torch

project_dir = Path('/content/cod-ssl')
if (project_dir / '.git').is_dir():
    subprocess.run(['git', '-C', str(project_dir), 'fetch', 'origin', PROJECT_BRANCH], check=True)
    subprocess.run(['git', '-C', str(project_dir), 'checkout', PROJECT_BRANCH], check=True)
    subprocess.run(['git', '-C', str(project_dir), 'pull', '--ff-only', 'origin', PROJECT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', PROJECT_BRANCH, PROJECT_REPO_URL, str(project_dir)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{project_dir}[dev,notebooks,vcod]'], check=True)

bootstrap_env = os.environ.copy()
dino_weights = Path(DRIVE_ROOT) / 'checkpoints/dinov3_vitb16.pth'
if not dino_weights.is_file():
    print('DINOv3 requires approved Meta access on the first run only.')
    private_url = getpass('Private DINOv3 ViT-B/16 LVD-1689M URL: ').strip()
    if not private_url: raise ValueError('The approved DINOv3 URL is required.')
    bootstrap_env['COD_SSL_DINOV3_DOWNLOAD_URL'] = private_url
    del private_url
state_file = Path('/content/cod_ssl_bootstrap_state.json')
subprocess.run([sys.executable, str(project_dir / 'scripts/bootstrap_colab.py'),
                '--project-dir', str(project_dir), '--drive-root', DRIVE_ROOT,
                '--state-file', str(state_file)],
               cwd=project_dir, env=bootstrap_env, check=True)
bootstrap_env.pop('COD_SSL_DINOV3_DOWNLOAD_URL', None)
state = json.loads(state_file.read_text())
os.environ.update(state['environment'])
PROJECT_DIR = Path(state['project_dir'])
VCOD_ROOT = Path(state['drive_root']) / 'vcod'
MOCA_MANIFEST = VCOD_ROOT / 'manifests/moca_mask.csv'
CAMOVID_MANIFEST = VCOD_ROOT / 'manifests/camovid60k.csv'
INSPECTION_ROOT = VCOD_ROOT / 'inspections'
APPROVAL_ROOT = VCOD_ROOT / 'approvals'
for path in (INSPECTION_ROOT, APPROVAL_ROOT): path.mkdir(parents=True, exist_ok=True)
os.environ['MOCA_MASK_MANIFEST'] = str(MOCA_MANIFEST)
os.environ['CAMOVID60K_MANIFEST'] = str(CAMOVID_MANIFEST)
os.chdir(PROJECT_DIR)
print('Ready on', state['gpu'], 'with VCOD root', VCOD_ROOT)

In [ ]:
# Runtime and asset preflight. The locked primary protocol uses BF16 and is intended for a high-memory GPU.
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime.')
properties = torch.cuda.get_device_properties(0)
gpu_report = {
    'name': properties.name,
    'memory_gib': round(properties.total_memory / 2**30, 2),
    'compute_capability': f'{properties.major}.{properties.minor}',
    'bf16_supported': torch.cuda.is_bf16_supported(),
}
print(json.dumps(gpu_report, indent=2))
if not gpu_report['bf16_supported']:
    raise RuntimeError('The locked BF16 protocol requires a BF16-capable GPU; request an A100-class runtime.')
for manifest in (MOCA_MANIFEST, CAMOVID_MANIFEST):
    if not manifest.is_file():
        raise FileNotFoundError(f'Place the canonical manifest at {manifest}')

In [ ]:
# Repository correctness gate before touching scientific data.
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=PROJECT_DIR, check=True)

In [ ]:
# Inspect all dataset/regime manifests. Reports and random overlays are persisted in Drive.
inspection_commands = [
    ('moca_mask', [sys.executable, 'scripts/inspect_dataset.py',
                   '--config', 'configs/datasets/moca_mask.yaml',
                   '--manifest', str(MOCA_MANIFEST),
                   '--output', str(INSPECTION_ROOT / 'moca_mask')]),
    ('camovid60k_small', [sys.executable, 'scripts/inspect_dataset.py',
                          '--config', 'configs/datasets/camovid60k.yaml',
                          '--manifest', str(CAMOVID_MANIFEST),
                          '--regime', 'small_displacement',
                          '--output', str(INSPECTION_ROOT / 'camovid60k_small')]),
    ('camovid60k_large', [sys.executable, 'scripts/inspect_dataset.py',
                          '--config', 'configs/datasets/camovid60k.yaml',
                          '--manifest', str(CAMOVID_MANIFEST),
                          '--regime', 'large_displacement',
                          '--output', str(INSPECTION_ROOT / 'camovid60k_large')]),
]
for name, command in inspection_commands:
    print('Inspecting', name)
    subprocess.run(command, cwd=PROJECT_DIR, check=True)
if hasattr(os, 'sync'): os.sync()

In [ ]:
# Audit source identities and frame names across datasets and CamoVid60K regimes.
import pandas as pd
from cod_ssl.utils.run import file_sha256
moca = pd.read_csv(MOCA_MANIFEST)
camovid = pd.read_csv(CAMOVID_MANIFEST)
small = camovid[camovid.regime.fillna('default') == 'small_displacement']
large = camovid[camovid.regime.fillna('default') == 'large_displacement']
small_sources, large_sources = set(small.source_video_id.astype(str)), set(large.source_video_id.astype(str))
common_frame_names = set(moca.frame_id.astype(str)) & set(camovid.frame_id.astype(str))
matching_hashes = []
for frame_name in sorted(common_frame_names):
    moca_paths = set(moca[moca.frame_id.astype(str) == frame_name].image_path.astype(str))
    camovid_paths = set(camovid[camovid.frame_id.astype(str) == frame_name].image_path.astype(str))
    moca_hashes = {file_sha256(path) for path in moca_paths if Path(path).is_file()}
    camovid_hashes = {file_sha256(path) for path in camovid_paths if Path(path).is_file()}
    if moca_hashes & camovid_hashes:
        matching_hashes.append({'frame_id': frame_name, 'sha256': sorted(moca_hashes & camovid_hashes)})
audit = {
    'moca_manifest_sha256': file_sha256(MOCA_MANIFEST),
    'camovid_manifest_sha256': file_sha256(CAMOVID_MANIFEST),
    'moca_camovid_source_overlap': sorted(set(moca.source_video_id.astype(str)) & set(camovid.source_video_id.astype(str))),
    'moca_camovid_frame_name_overlap_count': len(common_frame_names),
    'moca_camovid_matching_image_hashes': matching_hashes,
    'camovid_small_only_sources': sorted(small_sources - large_sources),
    'camovid_large_only_sources': sorted(large_sources - small_sources),
    'camovid_shared_source_count': len(small_sources & large_sources),
}
if audit['camovid_small_only_sources'] or audit['camovid_large_only_sources']:
    raise ValueError('CamoVid60K regimes do not preserve identical source-video identities.')
(INSPECTION_ROOT / 'cross_dataset_overlap.json').write_text(json.dumps(audit, indent=2) + '\n')
print(json.dumps(audit, indent=2))
if hasattr(os, 'sync'): os.sync()

In [ ]:
# Checkpoint-specific image/video pathway and dense-token mapping gate.
commands = [
    [sys.executable, 'scripts/inspect_backbone.py', '--model', 'dinov3_vitb16', '--pathway', 'image'],
    [sys.executable, 'scripts/inspect_backbone.py', '--model', 'vjepa21_vitb16', '--pathway', 'image'],
    [sys.executable, 'scripts/inspect_backbone.py', '--model', 'vjepa21_vitb16', '--pathway', 'video',
     '--clip-length', '64', '--target-index', '32'],
]
for command in commands:
    subprocess.run(command, cwd=PROJECT_DIR, check=True)
    gc = __import__('gc'); gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Display automated reports and overlay contact sheets for the required manual review.
from IPython.display import Markdown, display
from PIL import Image
for name in ('moca_mask', 'camovid60k_small', 'camovid60k_large'):
    directory = INSPECTION_ROOT / name
    display(Markdown((directory / 'report.md').read_text()))
    display(Image.open(directory / 'random_overlays.png'))

In [ ]:
# Inspect deterministic boundary padding and chronological source indices on real videos.
from cod_ssl.data.clip_sampler import ClipSampler, ClipSpec
from PIL import ImageDraw
def boundary_contact_sheet(frame, label):
    source_id = sorted(frame.source_video_id.astype(str).unique())[0]
    group = frame[frame.source_video_id.astype(str) == source_id].sort_values('frame_number').reset_index(drop=True)
    sampler, spec = ClipSampler(), ClipSpec(5, 1, 2)
    panels = []
    for target_position in (0, len(group) - 1):
        positions, valid = sampler.source_indices(group.frame_number.astype(int).tolist(), target_position, spec)
        for slot, (position, is_valid) in enumerate(zip(positions, valid.tolist())):
            row = group.iloc[position]
            with Image.open(row.image_path) as raw: image = raw.convert('RGB')
            image.thumbnail((180, 130))
            panel = Image.new('RGB', (190, 160), 'white'); panel.paste(image, ((190-image.width)//2, 0))
            ImageDraw.Draw(panel).text((4, 136), f'target={target_position} slot={slot} frame={row.frame_number} valid={is_valid}', fill='black')
            panels.append(panel)
    sheet = Image.new('RGB', (5 * 190, 2 * 160), 'white')
    for index, panel in enumerate(panels): sheet.paste(panel, ((index % 5) * 190, (index // 5) * 160))
    path = INSPECTION_ROOT / f'{label}_boundary_clips.png'; sheet.save(path)
    print(label, 'source_video_id=', source_id); display(sheet)
boundary_contact_sheet(moca[moca.split == 'train'], 'moca_mask')
boundary_contact_sheet(small[small.split == 'train'], 'camovid60k_small')
boundary_contact_sheet(large[large.split == 'train'], 'camovid60k_large')
if hasattr(os, 'sync'): os.sync()

In [ ]:
#@title Manual correctness sign-off (edit every field after inspecting the outputs)
REVIEWER = '' #@param {type:'string'}
DATASET_RELEASES = '' #@param {type:'string'}
OVERLAYS_ALIGNED = False #@param {type:'boolean'}
BOUNDARY_CLIPS_CORRECT = False #@param {type:'boolean'}
VJEPA_TUBELET_MAPPING_APPROVED = False #@param {type:'boolean'}
FEATURE_ORIENTATION_APPROVED = False #@param {type:'boolean'}
WARNINGS_ACKNOWLEDGED = False #@param {type:'boolean'}

checks = [OVERLAYS_ALIGNED, BOUNDARY_CLIPS_CORRECT, VJEPA_TUBELET_MAPPING_APPROVED,
          FEATURE_ORIENTATION_APPROVED, WARNINGS_ACKNOWLEDGED]
if not REVIEWER.strip() or not DATASET_RELEASES.strip() or not all(checks):
    raise PermissionError('Complete every manual sign-off field before authorizing training.')
from datetime import datetime, timezone
approval = {
    'reviewer': REVIEWER.strip(), 'dataset_releases': DATASET_RELEASES.strip(),
    'approved_at_utc': datetime.now(timezone.utc).isoformat(),
    'moca_manifest_sha256': file_sha256(MOCA_MANIFEST),
    'camovid_manifest_sha256': file_sha256(CAMOVID_MANIFEST),
    'overlays_aligned': OVERLAYS_ALIGNED, 'boundary_clips_correct': BOUNDARY_CLIPS_CORRECT,
    'vjepa_tubelet_mapping_approved': VJEPA_TUBELET_MAPPING_APPROVED,
    'feature_orientation_approved': FEATURE_ORIENTATION_APPROVED,
    'warnings_acknowledged': WARNINGS_ACKNOWLEDGED,
}
approval_path = APPROVAL_ROOT / 'vcod_validation_approval.json'
approval_path.write_text(json.dumps(approval, indent=2) + '\n')
if hasattr(os, 'sync'): os.sync()
print('Training gate approved:', approval_path)